# 1. JupyterAI and Hosted JupyterHub Integration (Instructor: Mahidhar Tatineni)

The primary access mechanism we support for NAIRR Pilot Classroom usage on NRP is via JupyterHub. In this part of the tutorial we introduce NRP’s hosted JupyterHub environment, which provides an accessible, browser-based interface for interactive computation. Participants will log in using institutional credentials via CILogon and spawn compute instances with GPU/CPU/FPGA resources as required. Within these notebooks, JupyterAI integration allows users to connect directly to NRP-hosted LLMs for assisted code generation and in-notebook, chatbot-like debugging and assistance. 

## Introduction to the National Research Platform (NRP)

The National Research Platform (NRP) is a partnership of more than 50 institutions, led by researchers and cyberinfrastructure professionals at UC San Diego, University of Nebraska-Lincoln, and the Massachusetts Green High Performance Computing Center (MGHPCC). The NRP provides an open, nationally distributed cyberinfrastructure built on a Kubernetes cluster named **Nautilus**.

Nautilus pools heterogeneous hardware components—spanning compute, storage, and specialized accelerators like NVIDIA GPUs and Qualcomm Cloud AI devices—from contributing partners into a unified computing framework. Researchers access these resources through namespaces, allocating storage, running persistent applications, or executing temporary batch jobs.

**You do NOT need to request NVIDIA GPUs or Qualcomm devices in order to complete the inference and LLM portions of this tutorial. Hardware accelerators will be used later through Kubernetes workloads.**

### Available Compute Resources

Nautilus features a wide variety of computational resources:
- Standard x86 CPUs and high-memory CPU nodes.
- Diverse **NVIDIA GPUs** (e.g., A10, A100, RTX 3090/4090, H100) accessible for demanding parallel computing tasks and machine learning.
- Advanced hardware accelerators like **Qualcomm Cloud AI 100 Ultra SoCs** natively mapped as standard Kubernetes resources via Device Plugins. 

### Storage and Namespaces

By default, your work executes within Kubernetes **Namespaces**. These virtual partitions securely isolate compute workloads and data allocation models across distinct projects. A typical workload utilizes **Persistent Volume Claims (PVCs)** built mostly upon Ceph instances distributed globally. This mechanism allows stateful data generation mapped safely against node eviction policies.

In [ ]:
# Run this cell once to apply the NRP Tutorial theme (optional)
from IPython.display import HTML, display
display(HTML("""
<style>
  /* NRP Tutorial theme — indigo / slate / teal */
  .jp-MarkdownCell .jp-Cell-inputWrapper,
  .text_cell .input_area {
    background: linear-gradient(90deg, #eef2ff 0%, #f5f3ff 10%, #fafafa 22%);
    border-left: 4px solid #6366f1;
    padding: 0.75em 0 0.75em 1em;
    margin: 0.5em 0;
    border-radius: 0 8px 8px 0;
    box-shadow: 0 1px 3px rgba(99, 102, 241, 0.08);
  }
  .jp-MarkdownCell h1, .text_cell_render h1 {
    color: #312e81;
    font-weight: 700;
    border-bottom: 2px solid #6366f1;
    padding-bottom: 0.45em;
    margin-top: 0;
    background: linear-gradient(90deg, #e0e7ff 0%, rgba(224, 231, 255, 0.4) 70%, transparent 100%);
    padding: 0.35em 0 0.45em 0.6em;
    margin: 0 -0.6em 0.35em -0.6em;
    border-radius: 6px 6px 0 0;
  }
  .jp-MarkdownCell h2, .text_cell_render h2 {
    color: #4338ca;
    font-weight: 600;
    margin-top: 1.25em;
    margin-bottom: 0.45em;
    padding-left: 0.5em;
    border-left: 4px solid #6366f1;
    background: linear-gradient(90deg, rgba(99, 102, 241, 0.06) 0%, transparent 100%);
    padding: 0.25em 0.5em 0.25em 0.6em;
    border-radius: 0 4px 4px 0;
  }
  .jp-MarkdownCell h3, .text_cell_render h3 {
    color: #0f766e;
    font-weight: 600;
    margin-top: 1em;
    padding-left: 0.45em;
    border-left: 4px solid #14b8a6;
    background: linear-gradient(90deg, rgba(20, 184, 166, 0.06) 0%, transparent 100%);
    padding: 0.2em 0.45em 0.2em 0.55em;
    border-radius: 0 4px 4px 0;
  }
  .jp-MarkdownCell p, .jp-MarkdownCell li, .text_cell_render p, .text_cell_render li {
    line-height: 1.7;
    color: #374151;
  }
  .jp-MarkdownCell a, .text_cell_render a {
    color: #4f46e5;
    font-weight: 500;
  }
  .jp-MarkdownCell a:hover, .text_cell_render a:hover {
    color: #4338ca;
    text-decoration: underline;
  }
  .jp-MarkdownCell code, .text_cell_render code {
    background: #e0e7ff;
    color: #3730a3;
    padding: 0.22em 0.5em;
    border-radius: 5px;
    font-size: 0.9em;
    border: 1px solid #c7d2fe;
  }
  .jp-MarkdownCell pre, .text_cell_render pre {
    background: #f8fafc;
    border: 1px solid #e2e8f0;
    border-left: 4px solid #6366f1;
    border-radius: 6px;
    padding: 0.8em 1em;
  }
  .jp-MarkdownCell strong, .text_cell_render strong {
    color: #1e293b;
  }
  .jp-MarkdownCell hr, .text_cell_render hr {
    border: none;
    border-top: 2px solid #cbd5e1;
    margin: 1.5em 0;
  }
  .jp-MarkdownCell ul, .text_cell_render ul {
    padding-left: 1.5em;
  }
  .jp-MarkdownCell li::marker, .text_cell_render li::marker {
    color: #6366f1;
  }
  .jp-MarkdownCell table, .text_cell_render table {
    border-collapse: collapse;
    border: 1px solid #e2e8f0;
    border-radius: 6px;
    overflow: hidden;
  }
  .jp-MarkdownCell th, .text_cell_render th {
    background: #eef2ff;
    color: #3730a3;
    padding: 0.5em 0.75em;
    text-align: left;
  }
  .jp-MarkdownCell td, .text_cell_render td {
    padding: 0.45em 0.75em;
    border-top: 1px solid #e2e8f0;
  }
</style>
"""))

### Hosted JupyterHub at a glance

NRP runs a [hosted JupyterHub](https://jupyterhub-west.nrp-nautilus.io) you can use with your institutional credentials (CILogon). After logging in, choose the hardware profile for your instance and start running notebooks—no Kubernetes required.

Your home directory (`/home/jovyan`) is a persistent volume, **5GB** by default; don’t fill it up or your next Jupyter session may not start. You can request more space or use [CephFS](https://nrp.ai/documentation/userdocs/storage/ceph) for larger or shared workloads. The server will shut down about **1 hour** after your browser disconnects, so keep a tab open or use a stable connection if you need long-running work. For available images and custom setups, see the [scientific images](https://nrp.ai/documentation/userdocs/running/sci-img/) guide and [TensorFlow with Jupyter](https://nrp.ai/documentation/userdocs/jupyter/jupyter-pod/). More detail: [JupyterHub service](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub-service/).

**Hands-on:** Open [JupyterHub](https://jupyterhub-west.nrp-nautilus.io), log in with CILogon, and spawn an instance with your chosen profile to start using the platform.

**Join NRP & hands-on support:** Use these links to [get started with NRP](https://nrp.ai/documentation/userdocs/start/getting-started/) and to [join NRP's Matrix chat](https://nrp.ai/contact/) for hands-on support during the tutorial.

**Related documentation:** [JupyterHub service](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub-service/) · [Using Nautilus](https://nrp.ai/documentation/userdocs/start/using-nautilus/)